# Sesión 1 · Práctica: ¿cuánta información hay en un texto y en un LLM?

**Teoría de la Información · Prof. Víctor Adrián Sosa Hernández**

En esta práctica se miden entropías de datos reales, paso a paso:

| Parte | Qué se hace |
|---|---|
| 0 | Se programan las dos fórmulas de la clase y se comprueban con ejemplos que ya se conocen |
| 1 | Entropía de las letras del español, usando *El Quijote* |
| 2 | Entropía de los pesos de un modelo de lenguaje real (SmolLM2-135M) |

**Vínculo con la presentación de la clase** ([abrir presentación](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html)): cada sección indica, en un recuadro 🔗, la lámina
donde se vio el tema.

| Parte | Temas de la clase que se aplican | Láminas |
|---|---|---|
| 0 | contenido de información, entropía | [lámina 15 · 15](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/4), [lámina 21 · 21](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/10), [lámina 22 · 22](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/11) |
| 1 | entropía, entropía conjunta, condicional y regla de la cadena | [lámina 29 · 29](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/18), [lámina 31 · 31](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/1), [lámina 37 · 37](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/7), [lámina 41 · 41](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/11) |
| 2 | los pesos del LLM: signo, exponente y mantisa | [lámina 9 · 9](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/7), [lámina 10 · 10](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/8), [lámina 49 · 49](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/4/1) a [lámina 52 · 52](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/4/4) |

**Cómo usar el notebook.** Las celdas se ejecutan en orden con `Shift + Enter`. Antes de cada celda de código hay
una explicación de lo que hace; conviene leerla primero y predecir el resultado. Las celdas marcadas con ✍️ son
ejercicios.

> **Si la red es lenta:** la parte 2 descarga un archivo de 270 MB. Puede ejecutarse primero la celda **2.3**
> (descarga) y, mientras termina, avanzar con la parte 1.

## Parte 0 · Las herramientas

### 0.1 Bibliotecas

Se usan cuatro bibliotecas de Python:

| Biblioteca | Para qué se usa |
|---|---|
| `numpy` | cálculos con arreglos grandes de números (millones de pesos a la vez) |
| `collections` | contar cuántas veces aparece cada letra (`Counter`) |
| `matplotlib` | gráficas |
| `unicodedata`, `re`, `json`, `struct`, `zlib`, `urllib` | limpiar texto, leer archivos binarios, comprimir y descargar |

In [ ]:
import numpy as np
import collections, unicodedata, re, json, struct, zlib, os, urllib.request
import matplotlib.pyplot as plt
print("Bibliotecas listas")

### 0.2 Contenido de información de un evento

> 🔗 **En la clase:** [lámina 15 · Definición: contenido de información](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/4) · [lámina 17 · ¿Qué significa un número de bits?](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/6)


Es la ecuación 1 de la presentación:

$$I(p) = -\log_2 p \quad \text{bits} \tag{1}$$

| Parte | Qué significa |
|---|---|
| $p$ | probabilidad del evento, entre 0 y 1 |
| $\log_2$ | logaritmo base 2: la unidad es el bit |
| $-$ | como $\log_2 p \le 0$, el signo menos lo vuelve positivo |
| $I(p)$ | contenido de información: cuánto informa enterarse de que el evento ocurrió |

La celda define la función y la prueba con los ejemplos de la clase. **Resultado esperado:** 0, 1, 2, 5.70 y 9.97 bits.

In [ ]:
def contenido_informacion(p):
    # Contenido de información, ecuación (1), en bits
    return -np.log2(p) + 0.0   # + 0.0 evita que se imprima "-0.00" cuando p = 1

ejemplos = [("sale el sol (seguro)", 1), ("una moneda cae cara", 1/2),
            ("dos monedas caen cara", 1/4), ("se adivina una carta", 1/52),
            ("gana un boleto entre 1000", 1/1000)]
for evento, p in ejemplos:
    print(f"{evento:28s} p = {p:<8.4g} I(p) = {contenido_informacion(p):5.2f} bits")

### 0.3 Entropía a partir de probabilidades

> 🔗 **En la clase:** [lámina 21 · Entropía](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/10) · [lámina 22 · Paso a paso: moneda cargada](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/11)


La entropía es el **promedio** de los contenidos de información, pesado por la probabilidad de cada valor
(ecuación 4 de la presentación):

$$H(X) = \sum_{x} p(x)\,\big(-\log_2 p(x)\big) \tag{4}$$

| Parte | Qué significa |
|---|---|
| $x$ | cada valor posible de la variable $X$ |
| $p(x)$ | probabilidad de ese valor |
| $-\log_2 p(x)$ | su contenido de información (ec. 1) |
| $p(x)\,(-\log_2 p(x))$ | lo que **aporta** ese valor al promedio (un "término") |
| $\sum_x$ | se suman los términos de todos los valores |

Primero se calcula **a mano**, término por término, para la moneda cargada de la clase ($0.9$ cara, $0.1$ cruz).
**Resultado esperado:** $0.137 + 0.332 = 0.469$ bits.

In [ ]:
probabilidades = {"cara": 0.9, "cruz": 0.1}

H = 0
for valor, p in probabilidades.items():
    I = contenido_informacion(p)
    termino = p * I
    H += termino
    print(f"{valor:5s}: p = {p}   I = {I:.3f} bits   término p·I = {termino:.3f}")
print(f"\nH = suma de los términos = {H:.3f} bits")

### 0.4 De conteos a probabilidades

> 🔗 **En la clase** las probabilidades eran dadas (por ejemplo, [lámina 32 · clima y tráfico](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/2)). Con datos reales hay que **estimarlas** contando.


Con datos reales no se conocen las probabilidades: se **estiman** contando. Si un valor aparece $n_x$ veces en
$N$ observaciones, su probabilidad estimada es su frecuencia relativa:

$$\hat p(x) = \frac{n_x}{N} \tag{A}$$

| Parte | Qué significa |
|---|---|
| $n_x$ | número de veces que aparece el valor $x$ |
| $N$ | total de observaciones, $N = \sum_x n_x$ |
| $\hat p(x)$ | probabilidad **estimada** (el sombrero indica que viene de los datos) |

La función `entropia` recibe conteos, los convierte en probabilidades con la ecuación (A) y aplica la ecuación (4).
Los valores con conteo 0 se descartan, porque por convención $0 \log_2 0 = 0$.

**Comprobación:** con conteos de una moneda justa (1, 1), una cargada (9, 1) y un dado (seis valores iguales) deben
salir 1, 0.469 y 2.585 bits.

In [ ]:
def entropia(conteos):
    # Entropía en bits a partir de conteos: ecuaciones (A) y (4)
    c = np.asarray(conteos, dtype=float)
    p = c[c > 0] / c.sum()          # ecuación (A), sin los valores que no aparecen
    return -(p * np.log2(p)).sum()   # ecuación (4)

print(f"moneda justa  : {entropia([1, 1]):.3f} bits")
print(f"moneda cargada: {entropia([9, 1]):.3f} bits")
print(f"dado justo    : {entropia([1]*6):.3f} bits")

## Parte 1 · Entropía del español

**Pregunta:** si se elige una letra al azar de un texto en español, ¿cuánta información trae en promedio? Con 27
símbolos posibles el máximo sería $\log_2 27 = 4.75$ bits, pero el español no usa todas las letras por igual.

### 1.1 Descargar *El Quijote*

Se descarga el texto completo desde el Proyecto Gutenberg (unos 2 MB). Si el archivo ya existe, no se vuelve a
descargar.

In [ ]:
URL_QUIJOTE = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
if not os.path.exists("quijote.txt"):
    urllib.request.urlretrieve(URL_QUIJOTE, "quijote.txt")

crudo = open("quijote.txt", encoding="utf-8").read()
print(f"{len(crudo):,} caracteres descargados")

### 1.2 Limpiar el texto

Para contar letras conviene un alfabeto pequeño y fijo. La limpieza tiene cinco pasos:

| Paso | Qué hace | Ejemplo |
|---|---|---|
| 1 | Recorta el encabezado y la licencia de Gutenberg | se queda desde "En un lugar de la Mancha" |
| 2 | Pasa todo a minúsculas | `Mancha` → `mancha` |
| 3 | Quita acentos y diéresis, **conservando la ñ** | `rocín` → `rocin`, `año` → `año` |
| 4 | Cambia todo lo que no sea letra, ñ o espacio por un espacio | `galgo corredor.` → `galgo corredor ` |
| 5 | Junta espacios repetidos en uno solo | `a  b` → `a b` |

El paso 3 usa la normalización Unicode NFD, que separa cada letra de su acento (`í` = `i` + acento); luego se borran
los acentos excepto la tilde de la ñ.

In [ ]:
t = crudo[crudo.find("En un lugar de la Mancha"): crudo.find("*** END")]      # paso 1
t = t.lower()                                                                  # paso 2
t = unicodedata.normalize("NFD", t)                                            # paso 3
t = "".join(c for c in t if unicodedata.category(c) != "Mn" or c == "̃")
t = unicodedata.normalize("NFC", t)
t = re.sub("[^a-zñ ]+", " ", t)                                                # paso 4
t = re.sub(" +", " ", t)                                                       # paso 5

print("Antes :", crudo[crudo.find("En un lugar"):][:120].replace("\n", " "))
print("Después:", t[:120])
print(f"\nTexto limpio: {len(t):,} caracteres")

### 1.3 Contar cada símbolo

`collections.Counter` recorre el texto y cuenta cuántas veces aparece cada símbolo: son los $n_x$ de la ecuación
(A). La tabla muestra los diez más frecuentes con su probabilidad estimada $\hat p = n_x / N$.

In [ ]:
conteo_letras = collections.Counter(t)
N = len(t)
print(f"Símbolos distintos: {len(conteo_letras)}     N = {N:,}\n")
print("símbolo   conteo      p̂")
for simbolo, n in conteo_letras.most_common(10):
    nombre = "espacio" if simbolo == " " else simbolo
    print(f"{nombre:8s} {n:9,d}   {n/N:.4f}")

Observaciones:

- Aparecen **27** símbolos: 26 letras más el espacio… menos una. En todo *El Quijote* no hay ninguna **k**.
- El espacio es el símbolo más frecuente (casi 1 de cada 5) y la **e** es la letra más común.

### 1.4 Entropía de una letra, término por término

> 🔗 **En la clase:** [lámina 22 · el mismo cálculo término por término con la moneda](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/11) · [lámina 29 · Propiedades básicas: el español mide 3.98 bits](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/18)


Se aplica la ecuación (4) a las probabilidades estimadas. Para ver el proceso, la celda imprime el término
$\hat p \cdot (-\log_2 \hat p)$ de los cinco símbolos más frecuentes antes de dar el total.

**Resultado esperado:** alrededor de 3.98 bits, frente al máximo $\log_2 27 = 4.755$.

In [ ]:
print("símbolo     p̂      I = -log2 p̂   término p̂·I")
for simbolo, n in conteo_letras.most_common(5):
    p = n / N
    nombre = "espacio" if simbolo == " " else simbolo
    print(f"{nombre:8s} {p:.4f}   {contenido_informacion(p):8.3f}      {p*contenido_informacion(p):.4f}")
print("  ... (y así con los 27 símbolos)\n")

H1 = entropia(list(conteo_letras.values()))
print(f"H(letra) = {H1:.3f} bits")
print(f"Máximo posible con {len(conteo_letras)} símbolos: log2({len(conteo_letras)}) = {np.log2(len(conteo_letras)):.3f} bits")

### 1.5 Gráfica de frecuencias

> 🔗 **En la clase:** [lámina 29 · cota $H \le \log_2 n$, máximo cuando todo es igual de probable](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/18)


Si todas las letras fueran igual de probables, todas las barras tendrían la misma altura ($1/27 \approx 0.037$, línea
punteada) y la entropía sería la máxima. La diferencia entre las barras y la línea es lo que hace que $H$ sea menor.

In [ ]:
orden = conteo_letras.most_common()
etiquetas = ["␣" if s == " " else s for s, _ in orden]
plt.figure(figsize=(10, 3))
plt.bar(etiquetas, [n / N for _, n in orden])
plt.axhline(1 / len(orden), ls="--", color="gray", label="si fueran uniformes")
plt.ylabel("p̂"); plt.title("Frecuencia de cada símbolo en El Quijote"); plt.legend()
plt.show()

### 1.6 Pares de letras: entropía conjunta

> 🔗 **En la clase:** [lámina 31 · Entropía conjunta](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/1) · [lámina 32 · Paso a paso: clima y tráfico](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/2)


Ahora la variable no es una letra sino un **par** de letras consecutivas $(X_1, X_2)$, por ejemplo `"qu"` o `"a "`.
Su entropía es la entropía conjunta (ecuación 8 de la presentación):

$$H(X_1, X_2) = -\sum_{x_1}\sum_{x_2} p(x_1, x_2)\,\log_2 p(x_1, x_2) \tag{8}$$

| Parte | Qué significa |
|---|---|
| $(x_1, x_2)$ | un par de letras consecutivas |
| $p(x_1, x_2)$ | probabilidad de ese par, estimada contando pares con la ecuación (A) |
| $H(X_1, X_2)$ | contenido de información promedio de un par completo |

`zip(t, t[1:])` recorre el texto de dos en dos: ("e","n"), ("n"," "), (" ","u"), …

In [ ]:
conteo_pares = collections.Counter(zip(t, t[1:]))
print("Pares más frecuentes:")
for (a, b), n in conteo_pares.most_common(8):
    print(f"  '{a}{b}'  {n:8,d}")
H12 = entropia(list(conteo_pares.values()))
print(f"\nPares distintos: {len(conteo_pares)}     H(X1, X2) = {H12:.3f} bits")

### 1.7 Entropía condicional: ¿cuánto informa una letra si ya se conoce la anterior?

> 🔗 **En la clase:** [lámina 37 · Entropía condicional](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/7) · [lámina 41 · Regla de la cadena](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/11) · [lámina 44 · ¿Qué pregunta responde cada medida?](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/14)


La regla de la cadena (ecuación 10 de la presentación) dice

$$H(X_1, X_2) = H(X_1) + H(X_2 \mid X_1) \tag{10}$$

y despejando se obtiene lo que se busca:

$$H(X_2 \mid X_1) = H(X_1, X_2) - H(X_1) \tag{B}$$

| Parte | Qué significa |
|---|---|
| $H(X_1, X_2)$ | información promedio del par (sección 1.6) |
| $H(X_1)$ | información promedio de una letra sola (sección 1.4) |
| $H(X_2 \mid X_1)$ | entropía condicional (ecuación 9): información que **queda** en la segunda letra cuando ya se conoce la primera |

**Resultado esperado:** alrededor de 3.04 bits, menos que los 3.98 de una letra sola.

In [ ]:
H2_dado_1 = H12 - H1     # ecuación (B)
print(f"H(X1, X2)       = {H12:.3f} bits")
print(f"H(X1)           = {H1:.3f} bits")
print(f"H(X2 | X1)      = {H12:.3f} - {H1:.3f} = {H2_dado_1:.3f} bits")
print(f"\nConocer la letra anterior ahorra {H1 - H2_dado_1:.3f} bits por letra.")

### 1.8 ✍️ Ejercicio: condicionar en **dos** letras anteriores

> 🔗 **En la clase:** es la situación del teclado de celular de [lámina 45 · ¿Cuál se usa?](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/3/15): adivinar lo que sigue conociendo lo anterior.


Se repite el proceso con **ternas** de letras $(X_1, X_2, X_3)$. Por la regla de la cadena,

$$H(X_3 \mid X_1, X_2) = H(X_1, X_2, X_3) - H(X_1, X_2) \tag{C}$$

| Parte | Qué significa |
|---|---|
| $H(X_1, X_2, X_3)$ | entropía conjunta de una terna; se cuentan ternas con `zip(t, t[1:], t[2:])` |
| $H(X_1, X_2)$ | la de los pares, ya calculada (`H12`) |
| $H(X_3 \mid X_1, X_2)$ | información que queda en una letra cuando se conocen las **dos** anteriores |

**Pasos:** (1) contar ternas con `collections.Counter`, (2) calcular su entropía con `entropia`, (3) aplicar la
ecuación (C).

In [ ]:
# ✍️ Paso 1: contar ternas
conteo_ternas = None

# ✍️ Paso 2: entropía de las ternas
H123 = None

# ✍️ Paso 3: ecuación (C)
H3_dado_12 = None

if H3_dado_12 is None:
    print("Falta completar los tres pasos")
else:
    print(f"H(X3 | X1, X2) = {H3_dado_12:.3f} bits")

**✍️ Preguntas**

1. ¿Por qué baja la entropía cada vez que se conoce más contexto? Dar un ejemplo de una letra que, después de otra,
   es casi segura.
2. Con contextos cada vez más largos, la estimación por conteo se vuelve demasiado optimista. ¿Por qué? (Pista:
   ¿cuántas veces aparece cada contexto largo en el texto?)

*Respuestas:*

## Parte 2 · ¿Cuánta información hay en un peso de un LLM?

### 2.1 El formato bfloat16

> 🔗 **En la clase:** [lámina 9 · Un adelanto: ¿cuántos bits necesita un peso?](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/7) · [lámina 49 · Cómo se lee un peso del LLM](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/4/1)


*SmolLM2-135M* es un modelo de lenguaje con 134.5 millones de pesos. Cada peso se guarda en 16 bits con el formato
**bfloat16**:

```
 ┌───┬───────────┬─────────┐
 │ s │ exponente │ mantisa │      1 + 8 + 7 = 16 bits
 └───┴───────────┴─────────┘
```

$$\text{valor} = (-1)^{s} \cdot 2^{\,e-127} \cdot \left(1 + \frac{m}{128}\right) \tag{12}$$

| Parte | Qué significa |
|---|---|
| $s$ | signo: 0 positivo, 1 negativo |
| $e$ | exponente guardado, entero de 0 a 255 |
| $e - 127$ | exponente **sin sesgo** $k$: la potencia de 2 que da el tamaño del número; para pesos menores que 1 es negativo |
| $m$ | mantisa, entero de 0 a 127 |
| $1 + m/128$ | factor entre 1 y 2 que ubica el número dentro de su potencia de 2 |

**Ejemplo:** $0.15$ está entre $2^{-3} = 0.125$ y $2^{-2} = 0.25$, así que $k = -3$, $e = 124$, y
$0.15 / 0.125 = 1.2 = 1 + m/128$ da $m \approx 26$.

### 2.2 El archivo `.safetensors`

Los pesos vienen en un archivo con una estructura muy simple:

```
 ┌──────────────┬───────────────────────────┬──────────────────────────────────┐
 │ 8 bytes:     │ encabezado JSON:          │ los pesos crudos, uno tras otro  │
 │ tamaño del   │ nombre, tipo y forma de   │ (134.5 millones × 2 bytes)       │
 │ encabezado   │ cada tensor               │                                  │
 └──────────────┴───────────────────────────┴──────────────────────────────────┘
```

No hace falta PyTorch: basta leer el encabezado y luego interpretar el resto como enteros de 16 bits.

### 2.3 Descargar el modelo (270 MB)

In [ ]:
URL_MODELO = "https://huggingface.co/HuggingFaceTB/SmolLM2-135M/resolve/main/model.safetensors"
ARCHIVO = "smol.safetensors"
if not os.path.exists(ARCHIVO):
    urllib.request.urlretrieve(URL_MODELO, ARCHIVO)
print(f"{os.path.getsize(ARCHIVO) / 1e6:.1f} MB")

### 2.4 Leer el encabezado

1. Los primeros 8 bytes son un entero (`struct.unpack("<Q", ...)`, formato *little-endian* de 64 bits) con el
   tamaño del encabezado.
2. Se leen esos bytes y se interpretan como JSON.
3. La celda muestra los primeros tensores: su nombre, su tipo y su forma (dimensiones de la matriz).

In [ ]:
with open(ARCHIVO, "rb") as f:
    n_encabezado = struct.unpack("<Q", f.read(8))[0]
    encabezado = json.loads(f.read(n_encabezado))
    crudo_pesos = f.read()

tensores = {k: v for k, v in encabezado.items() if k != "__metadata__"}
print(f"Encabezado: {n_encabezado:,} bytes · {len(tensores)} tensores · tipos: {set(v['dtype'] for v in tensores.values())}\n")
for nombre, info in list(tensores.items())[:5]:
    print(f"{nombre:45s} {info['dtype']}  forma {info['shape']}")
print("...")

### 2.5 Los pesos como enteros de 16 bits

`np.frombuffer(..., dtype="<u2")` interpreta los bytes como enteros sin signo de 16 bits, sin copiar nada: cada
entero **es** un peso con sus 16 bits tal cual.

In [ ]:
pesos = np.frombuffer(crudo_pesos, dtype="<u2")
print(f"{pesos.size:,} pesos  =  {pesos.size * 2 / 1e6:.1f} MB")

### 2.6 Separar los tres campos con operaciones de bits

> 🔗 **En la clase:** los dos primeros pesos que imprime esta celda son los mismos que se decodificaron a mano en [lámina 50 · el primer peso](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/4/2) y [lámina 52 · el segundo peso](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/4/4).


| Campo | Operación | Qué hace |
|---|---|---|
| signo | `pesos >> 15` | recorre 15 lugares a la derecha: queda sólo el bit más alto |
| exponente | `(pesos >> 7) & 0xFF` | recorre 7 lugares (quita la mantisa) y se queda con 8 bits (`0xFF` = `11111111`) |
| mantisa | `pesos & 0x7F` | se queda con los 7 bits más bajos (`0x7F` = `1111111`) |

Para ver que todo funciona, la celda toma los primeros 5 pesos, muestra sus 16 bits, los separa en campos, calcula
su valor con la ecuación (12) y lo compara con el valor que da la computadora.

In [ ]:
signo     = pesos >> 15
exponente = (pesos >> 7) & 0xFF
mantisa   = pesos & 0x7F

# Comprobación con los primeros 5 pesos
valor_real = (pesos[:5].astype(np.uint32) << 16).view(np.float32)   # bfloat16 = mitad alta de un float32
print("bits (s|exponente|mantisa)      s    e    k     m    valor con (12)    valor real")
for i in range(5):
    b = f"{pesos[i]:016b}"
    s, e, m = int(signo[i]), int(exponente[i]), int(mantisa[i])
    valor = (-1)**s * 2.0**(e - 127) * (1 + m / 128)
    print(f"{b[0]}|{b[1:9]}|{b[9:]}       {s}  {e:3d}  {e-127:4d}  {m:3d}   {valor:+.6f}       {valor_real[i]:+.6f}")

### 2.7 Entropía del signo

Se cuentan los ceros y los unos del bit de signo (`np.bincount` cuenta cuántas veces aparece cada entero) y se
aplica la ecuación (4) término por término. **Resultado esperado:** prácticamente 1 bit.

In [ ]:
conteo_signo = np.bincount(signo, minlength=2)
for valor, nombre in [(0, "positivo"), (1, "negativo")]:
    p = conteo_signo[valor] / pesos.size
    print(f"{nombre:9s} conteo = {conteo_signo[valor]:,}   p̂ = {p:.4f}   término = {p*contenido_informacion(p):.4f}")
H_signo = entropia(conteo_signo)
print(f"\nH(signo) = {H_signo:.3f} bits")

### 2.8 Entropía del exponente

> 🔗 **En la clase:** [lámina 9 · el exponente ocupa 8 bits pero trae 2.66](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/7) · [lámina 28 · preguntas promedio = entropía](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/2/17)


El exponente tiene 256 valores posibles. La tabla muestra los más frecuentes, **ordenados de más a menos
frecuente** (para la suma el orden no importa). La columna $k = e - 127$ es el exponente sin sesgo: un peso con
exponente $k$ cumple $2^k \le |w| < 2^{k+1}$.

**Resultado esperado:** 33 valores usados y $H \approx 2.66$ bits, aunque el campo ocupa 8.

In [ ]:
conteo_exp = np.bincount(exponente, minlength=256)
usados = np.nonzero(conteo_exp)[0]
print(f"Exponentes distintos usados: {len(usados)} de 256\n")
print("  k      e      conteo        p̂      -log2 p̂   término    |peso| entre")
for e in sorted(usados, key=lambda v: -conteo_exp[v])[:8]:
    p = conteo_exp[e] / pesos.size
    k = e - 127
    print(f"{k:4d}   {e:3d}  {conteo_exp[e]:11,d}   {p:.4f}   {contenido_informacion(p):6.3f}    {p*contenido_informacion(p):.3f}"
          f"     {2.0**k:.4g} y {2.0**(k+1):.4g}")
print("  ...")
H_exp = entropia(conteo_exp)
print(f"\nH(exponente) = {H_exp:.3f} bits   (ocupa 8 bits)")

In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(usados - 127, conteo_exp[usados] / pesos.size)
plt.xlabel("exponente sin sesgo k = e - 127"); plt.ylabel("p̂")
plt.title("Distribución del exponente: casi todo se concentra en k = -7 … -2")
plt.xlim(-15, 2); plt.show()

### 2.9 Entropía de la mantisa

La mantisa tiene 128 valores posibles. Si fuera uniforme, cada valor tendría probabilidad $1/128 = 0.0078$ y la
entropía sería el máximo, 7 bits. La gráfica muestra qué tan lejos está de eso.

**Resultado esperado:** $H \approx 6.97$ bits.

In [ ]:
conteo_man = np.bincount(mantisa, minlength=128)
p_man = conteo_man / pesos.size
H_man = entropia(conteo_man)
print(f"p̂ más grande: {p_man.max():.5f} (m = {p_man.argmax()})   p̂ más chica: {p_man.min():.5f} (m = {p_man.argmin()})   uniforme: {1/128:.5f}")
print(f"H(mantisa) = {H_man:.3f} bits   (ocupa 7 bits)")

plt.figure(figsize=(10, 2.8))
plt.plot(range(128), p_man, label="p̂ de cada mantisa")
plt.axhline(1/128, ls="--", color="gray", label="uniforme")
plt.xlabel("mantisa m"); plt.ylabel("p̂"); plt.legend(); plt.show()

Las mantisas pequeñas son casi el doble de frecuentes que las grandes: los pesos están repartidos de forma
aproximadamente uniforme en escala **logarítmica** (como en la ley de Benford), no en escala lineal.

### 2.10 Bits que realmente se necesitan por peso

> 🔗 **En la clase:** [lámina 9 · la tabla de los tres campos](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/7) · [lámina 10 · ¿y eso cuánto ahorra?](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/8)


Si cada campo se codifica de forma óptima y por separado, cada peso necesita en promedio la suma de las tres
entropías. Dividiendo entre los 16 bits que ocupa se obtiene el **límite** de compresión sin pérdida:

$$\text{límite} = \frac{H(\text{signo}) + H(\text{exponente}) + H(\text{mantisa})}{16} \tag{E}$$

| Parte | Qué significa |
|---|---|
| numerador | bits que **realmente se necesitan** en promedio por peso |
| $16$ | bits que **ocupa** cada peso en bfloat16 |
| límite | fracción del tamaño original a la que se puede reducir el archivo sin perder nada, codificando los campos por separado |

In [ ]:
total = H_signo + H_exp + H_man
print(f"{'campo':10s} {'ocupa':>6s} {'entropía':>9s}")
for nombre, ocupa, h in [("signo", 1, H_signo), ("exponente", 8, H_exp), ("mantisa", 7, H_man)]:
    print(f"{nombre:10s} {ocupa:6d} {h:9.3f}")
print(f"{'total':10s} {16:6d} {total:9.3f}")
print(f"\nLímite (E) = {total:.3f} / 16 = {total/16:.1%}  →  {pesos.size*2/1e6:.0f} MB podrían ser {pesos.size*2/1e6*total/16:.0f} MB")

### 2.11 ¿Y `zip`?

> 🔗 **En la clase:** [lámina 10 · por qué zip se queda en 80 %](https://vsosahdz.github.io/TeoriaInformacion/sesiones/sesion-01.html#/1/8)


`zlib` es el algoritmo de `zip` (Lempel-Ziv seguido de un código de Huffman). Trabaja sobre **bytes** y busca
secuencias repetidas. Se prueba con los primeros 40 MB (tarda unos segundos). La razón de compresión es

$$\text{razón} = \frac{\text{tamaño comprimido}}{\text{tamaño original}} \tag{F}$$

| Parte | Qué significa |
|---|---|
| tamaño comprimido | bytes que produce `zlib` |
| tamaño original | bytes de la muestra (40 MB) |
| razón | menor es mejor: 0.80 significa que el archivo queda en el 80 % de su tamaño |

In [ ]:
muestra = pesos[:20_000_000].tobytes()
comprimido = zlib.compress(muestra, 6)
razon = len(comprimido) / len(muestra)
print(f"original   : {len(muestra)/1e6:.1f} MB")
print(f"comprimido : {len(comprimido)/1e6:.1f} MB")
print(f"razón (F)  : {razon:.1%}   frente al límite (E) de {total/16:.1%}")

### ✍️ Preguntas (son el punto de partida del proyecto)

1. ¿Por qué el exponente tiene tan poca entropía comparado con sus 8 bits? Usar la gráfica de la sección 2.8.
2. ¿Por qué la mantisa queda tan cerca de su máximo de 7 bits?
3. `zlib` no llega al límite. ¿Qué "no ve" `zlib` que la entropía por campo sí ve? (Pista: ¿en qué byte queda cada
   campo?)
4. El límite (E) supone que los pesos son independientes entre sí y que los campos se codifican por separado.
   ¿Podría un compresor bajar todavía más? ¿Qué información se está ignorando?

*Respuestas:*

## Resumen de la práctica

| Medición | Valor aproximado |
|---|---|
| Entropía de una letra del español | 3.98 bits (máximo 4.75) |
| … conociendo la letra anterior | 3.04 bits |
| … conociendo las dos anteriores (ejercicio) | 2.58 bits |
| Entropía de un peso de SmolLM2-135M (por campos) | 10.63 bits de 16 |
| Límite de compresión sin pérdida por campos | 66.5 % |
| Lo que logra `zlib` | 79.8 % |